In [1]:
import json

from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from config import (
    CHUNKS_JSON_PATH,
    DEFAULT_EMBEDDING_MODEL_ID,
)

/tmp/ipykernel_4098/1596643607.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [2]:
chunks = json.loads(CHUNKS_JSON_PATH.read_text(encoding="utf-8"))
docs = [
    Document(
        page_content=chunk["text"],
        metadata={"doc_id": chunk["doc_id"], "source": chunk["source"]},
    )
    for chunk in chunks
]

In [3]:
# Embed
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=HuggingFaceEmbeddings(model_name=DEFAULT_EMBEDDING_MODEL_ID),
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

/home/tijers/.cache/pypoetry/virtualenvs/genai-study-0QOMwwXl-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2046.82it/s]


In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from genai_study.model_client import ModelClient
from genai_study.config import BASE_QUESTION

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

model_client = ModelClient.from_config()

generate_queries = (
    prompt_perspectives
    | model_client.as_runnable()
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1628.87it/s]


ValueError: Got device==0, device is required to be within [-1, 0)

In [ ]:
from langchain_core.load import dumps, loads


def get_unique_union(documents: list[list]):
    """Unique union of retrieved docs"""
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]


# Retrieve
retrieval_chain = generate_queries | retriever.map() | get_unique_union
retrieval_chain.invoke({"question": BASE_QUESTION})

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/tmp/ipykernel_3693/578653464.py:11: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


[Document(metadata={'source': 'Mistborn.txt', 'doc_id': 'Mistborn.txt::chunk-1808'}, page_content='Vin stood. Breeze patted his forehead with a handkerchief as he trotted his animal up beside her. He’d let his hair grow longer since she’d last seen him, and he kept it slicked back, its lower fringe tickling his collar. It wasn’t greying yet, though he was in his mid-forties. He wore no hat—it had probably blown free—but he had on one of his rich suits and silken vests. They were powdered with black ash from his hurried ride.\n\n“Ah, Vin, my dear,” Breeze said, breathing almost as deeply as his horse. “I must say, that was a timely arrival on your part. Impressively flamboyant as well. I do hate to force a rescue—but, well, if one is necessary, then it might as well happen with style.”'),
 Document(metadata={'doc_id': 'Mistborn.txt::chunk-1476', 'source': 'Mistborn.txt'}, page_content='There has to be another way! Vin thought with despair. There has to …\n\nAnd then she saw it sitting d

In [ ]:
import evaluation.evaluate as evaluation_module
import evaluation.strategies as strategies_module

Evaluator = evaluation_module.Evaluator
RunnableStrategy = strategies_module.RunnableStrategy

multi_query_runnable = generate_queries | retriever.map()
multi_query_strategy = RunnableStrategy(
    multi_query_runnable,
    input_mapper=lambda question: {"question": question},
)

evaluator = Evaluator(
    k_values=(1, 3, 5),
    experiment_name="retrieval_comparison",
    fusion_method="ordered",
    max_workers=10,
)
metrics = evaluator.evaluate(
    multi_query_strategy, experiment_name="multi_query_ordered"
)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

KeyboardInterrupt: 

In [ ]:
from evaluation.evaluate import Evaluator
from evaluation.strategies import RunnableStrategy

multi_query_runnable = generate_queries | retriever.map()
multi_query_strategy = RunnableStrategy(
    multi_query_runnable,
    input_mapper=lambda question: {"question": question},
)

evaluator = Evaluator(
    k_values=(1, 3, 5),
    experiment_name="retrieval_comparison",
    fusion_method="rrf",
    max_workers=10,
)
metrics = evaluator.evaluate(multi_query_strategy, experiment_name="multi_query_rrf")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/